In [1]:
from langchain_core.tools import tool
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from typing_extensions import TypedDict, Annotated
from typing import Literal
from langgraph.graph import StateGraph, START, END
import operator
from config.llm import llm


# ========== 2. 工具 ==========
@tool
def add(a: int, b: int) -> int:
    """Adds a and b.

    Args:
        a: First int
        b: Second int
    """
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: First int
        b: Second int
    """
    return a * b

@tool
def divide(a: int, b: int) -> float:
    """Divide a by b.

    Args:
        a: First int
        b: Second int
    """
    return a / b

tools = [add, multiply, divide]
tools_by_name = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

# ========== 3. 状态 ==========
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

# ========== 4. 节点 ==========
def llm_call(state: MessagesState):
    """大模型决定：直接回答还是调工具。"""
    return {
        "messages": [
            llm_with_tools.invoke(
                [SystemMessage(content="你是一个计算助手，使用提供的工具来完成计算任务。")]
                + state["messages"]
            )
        ]
    }

def tool_node(state: MessagesState):
    """执行工具调用。"""
    result = []
    for tool_call in state["messages"][-1].tool_calls:
        t = tools_by_name[tool_call["name"]]
        observation = t.invoke(tool_call["args"])
        result.append(
            ToolMessage(content=str(observation), tool_call_id=tool_call["id"])
        )
    return {"messages": result}

# ========== 5. 路由 ==========
def should_continue(state: MessagesState) -> Literal["tool_node", "__end__"]:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tool_node"
    return END

# ========== 6. 组装图 ==========
graph_builder = StateGraph(MessagesState)

graph_builder.add_node("llm_call", llm_call)
graph_builder.add_node("tool_node", tool_node)

graph_builder.add_edge(START, "llm_call")
graph_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    ["tool_node", END]
)
graph_builder.add_edge("tool_node", "llm_call")

agent = graph_builder.compile()

# ========== 7. 测试 ==========
response = agent.invoke(
    {"messages": [HumanMessage(content="先算 3 加 4，再把结果乘以 2")]}
)
for msg in response["messages"]:
    msg.pretty_print()

/home/jiangtong/桌面/agent/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3701: UserWarning: WARNING! openai_api_key is not default parameter.
                openai_api_key was transferred to model_kwargs.
                Please confirm that openai_api_key is what you intended.
  exec(code_obj, self.user_global_ns, self.user_ns)
/home/jiangtong/桌面/agent/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3701: UserWarning: WARNING! openai_api_base is not default parameter.
                openai_api_base was transferred to model_kwargs.
                Please confirm that openai_api_base is what you intended.
  exec(code_obj, self.user_global_ns, self.user_ns)


================================ Human Message =================================

先算 3 加 4，再把结果乘以 2
================================== Ai Message ==================================

我来帮你完成这个计算。首先计算 3 加 4：
Tool Calls:
  add (call_-7833988187188882738)
 Call ID: call_-7833988187188882738
  Args:
    a: 3
    b: 4
================================= Tool Message =================================

7
================================== Ai Message ==================================
Tool Calls:
  multiply (call_-7833959428087866417)
 Call ID: call_-7833959428087866417
  Args:
    a: 7
    b: 2
================================= Tool Message =================================

14
================================== Ai Message ==================================

计算完成！

第一步：3 + 4 = 7

第二步：7 × 2 = 14

最终结果是 **14**。
